# Fase 1 — Inventario y perfil de los documentos aceptados

*Hábitos musicales, estado de ánimo, estrés académico y uso de IA en estudiantes de MAC, FES Acatlán*

Este cuaderno consolida en código el resultado del Momento A (`00_momento_A.ipynb`). Lee únicamente los tres documentos aceptados, ubicados en `data/raw/`, y describe su estructura y calidad. También establece el **diccionario de variables** que guiará la limpieza de la Fase 3. Los tres documentos rechazados se trasladaron a `data/excluded/` y no se usan a partir de esta fase. En este cuaderno no se transforma ningún dato.

Para proteger el anonimato, ninguna celda muestra números de cuenta ni respuestas individuales; solo se presentan conteos y estadísticos agregados.

In [1]:
# Configuración
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

np.random.seed(42)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from datos import ACEPTADOS, EXCLUDED, RAW, cargar, perfil_columnas, perfil_documento
from diccionario import DICCIONARIO

pd.set_option("display.max_colwidth", 90)
datos = {d: cargar(d) for d in ACEPTADOS}   # columnas renombradas según el diccionario

## 1. Inventario de fuentes

La Tabla 1 lista los documentos que se integrarán al estudio, con el rol que se les asignó en el Momento A. La celda también verifica que los rechazados ya no estén en la carpeta de datos crudos.

In [2]:
# Tabla 1. Inventario de los documentos aceptados
inventario = pd.DataFrame([
    {"doc": d, "documento": m["nombre"], "rol": m["rol"], "prefijo": m["prefijo"],
     "formato": Path(m["archivo"]).suffix, "en_data_raw": (RAW / m["archivo"]).exists()}
    for d, m in ACEPTADOS.items()
]).set_index("doc")
print("Archivos en data/raw:", len([p for p in RAW.iterdir() if p.name != ".gitkeep"]),
      "| en data/excluded:", len([p for p in EXCLUDED.iterdir() if p.name != ".gitkeep"]))
inventario

Archivos en data/raw: 4 | en data/excluded: 3


,documento,rol,prefijo,formato,en_data_raw
doc,,,,,
1,Música y estado de ánimo,Principal,m_,.xlsx,True
2,Entorno universitario y bienestar,Complementario fuerte,b_,.csv,True
3,Desempeño académico por el uso de IA,Complementario medio,ia_,.csv,True


## 2. Perfil estructural

La Tabla 2 resume, para cada documento, el número de registros y de variables, la cantidad de variables de texto, las celdas vacías, los duplicados (por fila completa y por número de cuenta), las cuentas con longitud anómala y el periodo de recolección.

In [3]:
# Tabla 2. Perfil estructural de cada documento aceptado
resumen = pd.DataFrame([perfil_documento(d, df) for d, df in datos.items()]).set_index("doc")
resumen.T

doc,1,2,3
documento,Música y estado de ánimo,Entorno universitario y bienestar,Desempeño académico por el uso de IA
rol,Principal,Complementario fuerte,Complementario medio
registros,24,19,24
variables,18,17,18
vars_texto,15,16,17
celdas_nulas,1,0,2
filas_duplicadas,0,0,0
cuentas_duplicadas,0,0,0
cuentas_no_9_digitos,1,0,0
fecha_min,2026-10-29,2026-11-01,2026-10-30


Los tres documentos tienen una estructura sencilla: una fila por estudiante, sin filas ni cuentas duplicadas y con muy pocos valores faltantes (tres celdas en total). El único problema de integridad de la llave es una cuenta de diez dígitos en el Documento 1, que impedirá vincular ese registro. Las respuestas se recolectaron entre el 29 de octubre y el 11 de noviembre de 2026.

## 3. Perfil por variable

Para cada variable se reportan el tipo con que `pandas` la interpreta, los valores faltantes, la cardinalidad absoluta y relativa y la longitud media de la respuesta en palabras. Una cardinalidad cercana al 100 % indica que casi cada estudiante respondió con una redacción distinta, es decir, que la variable es texto libre y requiere codificación.

In [4]:
# Tabla 3. Perfil por variable (una tabla por documento)
for d, df in datos.items():
    display(Markdown(f"**Documento {d}. {ACEPTADOS[d]['nombre']}**"))
    display(perfil_columnas(df))

**Documento 1. Música y estado de ánimo**

,tipo_pandas,nulos,pct_nulos,unicos,pct_unicos,palabras_prom
marca_temporal,datetime64[us],0,0.0,24,100.0,NaN
cuenta,Int64,0,0.0,24,100.0,NaN
m_genero_frecuente,str,0,0.0,24,100.0,8.5
m_horas_dia,object,0,0.0,23,95.8,3.7
m_momentos,str,1,4.2,23,95.8,12.5
m_plataforma,str,0,0.0,21,87.5,3.7
m_animo_antes_despues,str,0,0.0,24,100.0,9.2
m_generos_emocion,str,0,0.0,24,100.0,9.5
m_experiencia,str,0,0.0,24,100.0,16.6
m_musica_estres,str,0,0.0,23,95.8,13.7


**Documento 2. Entorno universitario y bienestar**

,tipo_pandas,nulos,pct_nulos,unicos,pct_unicos,palabras_prom
marca_temporal,str,0,0.0,19,100.0,NaN
cuenta,Int64,0,0.0,19,100.0,NaN
b_carrera,str,0,0.0,10,52.6,2.7
b_semestre,str,0,0.0,11,57.9,1.3
b_edad,str,0,0.0,9,47.4,1.2
b_convivencia,str,0,0.0,7,36.8,1.1
b_traslado_min,str,0,0.0,16,84.2,2.3
b_ambiente_hogar,str,0,0.0,18,94.7,3.2
b_sueno_calidad,str,0,0.0,16,84.2,5.3
b_espacio_estudio,str,0,0.0,19,100.0,5.1


**Documento 3. Desempeño académico por el uso de IA**

,tipo_pandas,nulos,pct_nulos,unicos,pct_unicos,palabras_prom
marca_temporal,str,0,0.0,24,100.0,NaN
cuenta,Int64,0,0.0,24,100.0,NaN
ia_herramientas,str,0,0.0,23,95.8,6.2
ia_fase_uso,str,0,0.0,23,95.8,12.2
ia_cambio_estudio,str,0,0.0,23,95.8,12.2
ia_verificacion,str,0,0.0,24,100.0,15.3
ia_impacto_tiempo,str,0,0.0,23,95.8,12.0
ia_calificaciones,str,1,4.2,22,91.7,8.8
ia_independencia,str,0,0.0,24,100.0,23.3
ia_estres,str,0,0.0,24,100.0,13.8


In [5]:
# Resumen: proporción de variables de contenido con alta cardinalidad (> 80 % de valores únicos)
contenido = {d: perfil_columnas(df.drop(columns=["marca_temporal", "cuenta"])) for d, df in datos.items()}
pd.DataFrame({
    "variables_contenido": {d: len(p) for d, p in contenido.items()},
    "alta_cardinalidad": {d: int((p["pct_unicos"] > 80).sum()) for d, p in contenido.items()},
    "palabras_prom_doc": {d: round(p["palabras_prom"].mean(), 1) for d, p in contenido.items()},
}).rename_axis("doc")

,variables_contenido,alta_cardinalidad,palabras_prom_doc
doc,,,
1,16,16,10.0
2,15,11,4.6
3,16,16,11.3


## 4. Compatibilidad de la llave entre los documentos aceptados

Antes de la integración (Fase 4) se verifica cuántos estudiantes aparecen en cada combinación de documentos. Esto permite anticipar cuántos registros se perderán con una unión interna.

In [6]:
# Tabla 4. Cobertura de la llave
conj = {d: set(df["cuenta"].dropna()) for d, df in datos.items()}
nucleo = conj[1] & conj[2] & conj[3]
cobertura = pd.DataFrame({
    "registros": {d: len(conj[d]) for d in conj},
    "en_nucleo_1∩2∩3": {d: len(conj[d] & nucleo) for d in conj},
    "fuera_del_nucleo": {d: len(conj[d] - nucleo) for d in conj},
}).rename_axis("doc")
print("Estudiantes en los tres documentos:", len(nucleo))
print("Doc 1 ∩ Doc 3 (sin Doc 2):", len(conj[1] & conj[3]))
cobertura

Estudiantes en los tres documentos: 19
Doc 1 ∩ Doc 3 (sin Doc 2): 23


,registros,en_nucleo_1∩2∩3,fuera_del_nucleo
doc,,,
1,24,19,5
2,19,19,0
3,24,19,5


## 5. Diccionario de variables

El diccionario asigna a cada columna un nombre corto con prefijo por documento (`m_` música, `b_` bienestar, `ia_` inteligencia artificial). Para cada variable registra la forma de respuesta, la codificación prevista para la Fase 3 y su uso en el análisis: hipótesis H1 a H4 (ver `docs/negocio.md`), indicadores clave o descripción. El diccionario se guarda en `docs/diccionario_variables.csv`.

In [7]:
# Tabla 5. Diccionario de variables (se exporta a docs/diccionario_variables.csv)
DICCIONARIO.to_csv(RAIZ / "docs" / "diccionario_variables.csv", index=False, encoding="utf-8-sig")
DICCIONARIO

,doc,pregunta_n,variable,pregunta,respuesta,codificacion,uso
0,1,-,marca_temporal,Marca temporal,Fecha y hora,datetime,Control
1,1,-,cuenta,Número de cuenta,Entero,Llave; se anonimiza con hash,Llave
2,1,P1,m_genero_frecuente,Tipo de música que escucha con mayor frecuencia y por qué,Texto libre,Grupo de género musical (palabras clave),"H3, KPI géneros, filtro"
3,1,P2,m_horas_dia,Tiempo diario dedicado a escuchar música,Numérica en texto,Horas (float) con regex,"H2, KPI horas"
4,1,P3,m_momentos,Momentos del día y actividades mientras escucha,Multirrespuesta en texto,"Banderas: estudio, traslado, ejercicio, descanso, trabajo",H2
5,1,P4,m_plataforma,Plataformas o dispositivos de escucha,Multirrespuesta en texto,"Banderas: Spotify, YouTube, otras",Descriptivo
6,1,P5,m_animo_antes_despues,Estado de ánimo antes y después de escuchar música favorita,Texto libre,Etiqueta binaria mejora_animo (sí/no),"H1-H4, variable objetivo, KPI % mejora"
7,1,P6,m_generos_emocion,"Géneros que lo hacen sentir feliz, tranquilo, triste o motivado",Texto libre,Banderas de género,H3
8,1,P7,m_experiencia,Experiencia en que una canción cambió su ánimo,Texto libre,Bandera: reporta experiencia (sí/no),Descriptivo
9,1,P8,m_musica_estres,Música que escucha cuando está triste o estresado,Texto libre,Grupo de género / tipo de uso,Indicador indirecto de estrés


In [8]:
# Tabla 6. Variables por forma de respuesta y documento (excluye marca temporal y cuenta)
contenido_dic = DICCIONARIO[~DICCIONARIO["variable"].isin(["marca_temporal", "cuenta"])]
pd.crosstab(contenido_dic["respuesta"], contenido_dic["doc"], margins=True, margins_name="Total")

doc,1,2,3,Total
respuesta,,,,
Categórica en texto,0,2,1,3
Likert en texto,0,2,0,2
Lista en texto,1,0,0,1
Multirrespuesta en texto,3,0,2,5
Numérica en texto,1,3,2,6
Ordinal en texto,4,7,0,11
Sí/No con explicación,1,0,2,3
Texto libre,6,1,9,16
Total,16,15,16,47


In [9]:
# Tabla 7. Variables que operacionalizan cada hipótesis
filas = []
for h in ["H1", "H2", "H3", "H4"]:
    v = contenido_dic[contenido_dic["uso"].str.contains(h)]["variable"].tolist()
    filas.append({"hipótesis": h, "n_variables": len(v), "variables": ", ".join(v)})
pd.DataFrame(filas).set_index("hipótesis")

,n_variables,variables
hipótesis,,
H1,4,"m_animo_antes_despues, m_maneja_emociones, b_desanimo, b_irritabilidad"
H2,7,"m_horas_dia, m_momentos, m_energia, m_concentracion, m_sueno, b_sueno_calidad, b_conce..."
H3,2,"m_genero_frecuente, m_generos_emocion"
H4,5,"m_animo_antes_despues, ia_independencia, ia_estres, ia_horas_dia, ia_guia_vs_atajo"


## 6. Conclusiones de la fase

1. Los tres documentos aceptados están en `data/raw/` en su forma original. Los rechazados quedaron aislados en `data/excluded/`, y ninguna de las dos carpetas se versiona en Git.
2. Los datos no tienen duplicados y casi no tienen valores faltantes. La principal dificultad es la **codificación del texto libre**: la mayoría de las variables de contenido tienen una redacción distinta en cada respuesta.
3. Se identificó una cuenta con formato inválido en el Documento 1 y se confirmó que el núcleo integrable es de 19 estudiantes.
4. El diccionario de variables vincula cada pregunta con una regla de codificación y con las hipótesis. Así, la limpieza de la Fase 3 se limitará a las variables necesarias para responder la pregunta de investigación.